# Emotion Annotation: Qualitative Error Analysis (LLM vs. Human)

**Data:** `ablations/sample_emo_annot.csv` (100 posts)
- L = LLM
- R = Human A
- G = Human B

We analyze: 
1. Disagreement distribution.
2. Qualitative examples of disagreements.
3. **Emotion Co-occurrence / Confusion Analysis**: When the LLM makes an error on one emotion, which other emotions were actually present according to humans?

In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_colwidth', 300)
df = pd.read_csv('../ablations/sample_emo_annot.csv')
emotions = ['anger', 'fear', 'sad', 'disgust']
print(f'Loaded {len(df)} annotated posts.')

Loaded 100 annotated posts.


In [2]:
rows = []
for emo in emotions:
    llm_col = emo + ' (L)'
    ha_col  = emo + ' (R)'
    hb_col  = emo + ' (G)'
    llm_fp  = int(((df[llm_col]==1) & (df[ha_col]==0) & (df[hb_col]==0)).sum())
    llm_fn  = int(((df[llm_col]==0) & (df[ha_col]==1) & (df[hb_col]==1)).sum())
    h_disag = int((df[ha_col] != df[hb_col]).sum())
    agree   = int(((df[llm_col]==df[ha_col]) & (df[llm_col]==df[hb_col])).sum())
    rows.append({'Emotion': emo.capitalize(), 'All Agree': agree,
                 'LLM FP (both humans=0)': llm_fp,
                 'LLM FN (both humans=1)': llm_fn,
                 'Human A!=B': h_disag})
summary = pd.DataFrame(rows).set_index('Emotion')
print('=== Overall Disagreement Summary ===')
print(summary.to_string())


=== Overall Disagreement Summary ===
         All Agree  LLM FP (both humans=0)  LLM FN (both humans=1)  Human A!=B
Emotion                                                                       
Anger           67                      15                       0          18
Fear            80                       5                       1          14
Sad             90                       2                       0           8
Disgust         74                       2                       2          22


In [3]:
# ── Emotion Co-occurrence Analysis ────────────────────────────────────────
# Define the gold label as majority vote of Human A and B (R and G)
for emo in emotions:
    df[f'{emo}_gold'] = ((df[f'{emo} (R)'] + df[f'{emo} (G)']) >= 1).astype(int)

print('=== EMOTION CO-OCCURRENCE / ERROR CONFUSION ANALYSIS ===\n')

for emo in emotions:
    # LLM FP: LLM = 1, Gold = 0
    fp_mask = (df[f'{emo} (L)'] == 1) & (df[f'{emo}_gold'] == 0)
    fp_df = df[fp_mask]
    if len(fp_df) > 0:
        print(f'When LLM had a False Positive for {emo.upper()} (n={len(fp_df)} posts):')
        print('  Which gold emotions were actually present according to humans?')
        for other in emotions:
            if other == emo: continue
            pct = fp_df[f'{other}_gold'].mean() * 100
            print(f'    - {other.capitalize()}: {pct:.1f}% of the time')
        print()

for emo in emotions:
    # LLM FN: LLM = 0, Gold = 1
    fn_mask = (df[f'{emo} (L)'] == 0) & (df[f'{emo}_gold'] == 1)
    fn_df = df[fn_mask]
    if len(fn_df) > 0:
        print(f'When LLM had a False Negative for {emo.upper()} (n={fn_df.shape[0]} posts):')
        print('  Which other emotions did the LLM predict (LLM = 1)?')
        for other in emotions:
            if other == emo: continue
            pct = fn_df[f'{other} (L)'].mean() * 100
            print(f'    - {other.capitalize()}: {pct:.1f}% of the time')
        print()


=== EMOTION CO-OCCURRENCE / ERROR CONFUSION ANALYSIS ===

When LLM had a False Positive for ANGER (n=15 posts):
  Which gold emotions were actually present according to humans?
    - Fear: 60.0% of the time
    - Sad: 93.3% of the time
    - Disgust: 60.0% of the time

When LLM had a False Positive for FEAR (n=5 posts):
  Which gold emotions were actually present according to humans?
    - Anger: 60.0% of the time
    - Sad: 80.0% of the time
    - Disgust: 80.0% of the time

When LLM had a False Positive for SAD (n=2 posts):
  Which gold emotions were actually present according to humans?
    - Anger: 50.0% of the time
    - Fear: 50.0% of the time
    - Disgust: 100.0% of the time

When LLM had a False Positive for DISGUST (n=2 posts):
  Which gold emotions were actually present according to humans?
    - Anger: 50.0% of the time
    - Fear: 0.0% of the time
    - Sad: 100.0% of the time

When LLM had a False Negative for ANGER (n=3 posts):
  Which other emotions did the LLM predict 